# PCA LGBM Experiment

Test whether PCA components can replace the original processed feature set. `StandardScaler` and `PCA` are fitted inside each CV fold to avoid leakage.

In [1]:
# Add the project root to Python path so notebook imports can see the local src package.
import sys

sys.path.append("../")

# NumPy is used for log-transforming the target and aggregating CV scores.
import numpy as np
# Pandas is used for tabular data manipulation and readable result tables.
import pandas as pd

In [2]:
# PCA compresses many original columns into a smaller set of orthogonal components.
from sklearn.decomposition import PCA
# Metrics are calculated on original target scale after inverse log transform.
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
# train_test_split creates the held-out test split; KFold drives cross-validation.
from sklearn.model_selection import KFold, train_test_split
# PCA is sensitive to feature scale, so features are standardized before decomposition.
from sklearn.preprocessing import StandardScaler

# Project helpers keep loading, formatting, and model construction consistent across notebooks.
from src.loader import Loader
from src.metrics import format_metric_value
from src.modeling import build_lgbm_regressor

In [3]:
# Fixed seed makes the split, CV folds, and randomized PCA reproducible.
SEED = 42
# Keep one third of the data as a final sanity-check holdout set.
TEST_SIZE = 0.33
# Use 5-fold CV to choose the PCA component count on the training split only.
CV = 5

# Candidate dimensionalities: each value means the model sees only this many PCA features.
PCA_COMPONENTS = [10, 25, 50, 100]

## Load Data

In [4]:
# Load the already processed modeling table produced by earlier notebooks.
loader = Loader()
df = loader.load("../data/processed_data.csv")
# Show rows and columns as a quick sanity check.
df.shape

(4459, 4732)

In [5]:
# All columns except target are input features.
X = df.drop(columns="target")
# Keep the raw target for final metric calculation on the original scale.
y = df["target"]
# Train LightGBM on log1p(target); RMSE in this space corresponds to RMSLE.
y_log = np.log1p(y)

# Confirm the feature matrix and target vector dimensions.
(X.shape, y.shape)

((4459, 4731), (4459,))

## Fixed Data Setup

In [6]:
# Split both raw target and log target so each is available later.
X_train_raw, X_test_raw, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

# Shuffle folds so each fold has a more representative mix of rows.
cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

## Model Setup

Use a fixed conservative `LGBMRegressor` setup so this notebook measures the PCA feature effect, not another hyperparameter search.

In [7]:
# Fixed LightGBM parameters: PCA_COMPONENTS is the variable being tested here.
LGBM_PARAMS = {
    # Number of boosting rounds.
    "n_estimators": 700,
    # Small learning rate for a smoother, more conservative fit.
    "learning_rate": 0.01,
    # Tree complexity controls.
    "num_leaves": 31,
    "max_depth": 6,
    "min_child_samples": 40,
    # Row and column sampling reduce variance and overfitting risk.
    "subsample": 0.75,
    "subsample_freq": 1,
    "colsample_bytree": 0.75,
    # Regularization terms penalize overly complex trees.
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "min_split_gain": 0.05,
}

## PCA Helpers

In [8]:
def build_pca_features(
    X_fit: pd.DataFrame,
    X_transform: pd.DataFrame,
    n_components: int,
) -> tuple[pd.DataFrame, pd.DataFrame, float]:
    # Fit the scaler only on X_fit to avoid leaking information from validation/test rows.
    scaler = StandardScaler()
    X_fit_scaled = scaler.fit_transform(X_fit)
    # Apply the already-fitted scaler to the second matrix.
    X_transform_scaled = scaler.transform(X_transform)

    # Fit PCA only on X_fit; randomized solver is faster for wide feature matrices.
    pca = PCA(n_components=n_components, svd_solver="randomized", random_state=SEED)
    X_fit_pca = pca.fit_transform(X_fit_scaled)
    # Transform validation/test rows using the PCA basis learned from X_fit.
    X_transform_pca = pca.transform(X_transform_scaled)

    # Give components stable names so downstream tables are readable.
    pca_columns = [f"pca_{idx + 1:03d}" for idx in range(n_components)]
    # Convert arrays back to DataFrames and preserve original indexes for alignment.
    X_fit_pca = pd.DataFrame(X_fit_pca, columns=pca_columns, index=X_fit.index)
    X_transform_pca = pd.DataFrame(X_transform_pca, columns=pca_columns, index=X_transform.index)

    # Return both transformed matrices plus total explained variance for diagnostics.
    return X_fit_pca, X_transform_pca, float(pca.explained_variance_ratio_.sum())

In [9]:
def evaluate_cv(n_components: int) -> dict:
    # Store one validation score and one explained-variance value per fold.
    fold_scores = []
    explained_variance = []

    # CV is run only on the training split; the test split remains untouched until the end.
    for fold_train_idx, fold_valid_idx in cv.split(X_train_raw, y_train_log):
        # Select fold-specific train and validation rows.
        X_fold_train_raw = X_train_raw.iloc[fold_train_idx]
        X_fold_valid_raw = X_train_raw.iloc[fold_valid_idx]
        y_fold_train_log = y_train_log.iloc[fold_train_idx]
        y_fold_valid_log = y_train_log.iloc[fold_valid_idx]

        # Fit scaler/PCA on the fold train part and transform the fold validation part.
        X_fold_train, X_fold_valid, variance = build_pca_features(
            X_fold_train_raw,
            X_fold_valid_raw,
            n_components=n_components,
        )

        # Train LightGBM on PCA components only.
        model = build_lgbm_regressor(LGBM_PARAMS)
        model.fit(X_fold_train, y_fold_train_log)
        y_fold_pred_log = model.predict(X_fold_valid)

        # Log-space RMSE is the CV optimization metric for the log1p target setup.
        fold_scores.append(root_mean_squared_error(y_fold_valid_log, y_fold_pred_log))
        explained_variance.append(variance)

    # Return aggregate CV diagnostics for this component count.
    return {
        "n_components": n_components,
        "cv_rmsle_mean": float(np.mean(fold_scores)),
        "cv_rmsle_std": float(np.std(fold_scores)),
        "explained_variance_mean": float(np.mean(explained_variance)),
    }

## CV Comparison

In [10]:
# Collect CV results for every candidate number of PCA components.
results = []

for n_components in PCA_COMPONENTS:
    results.append(evaluate_cv(n_components=n_components))

# Sort by mean CV RMSLE; the first row is the best candidate.
results_df = pd.DataFrame(results).sort_values("cv_rmsle_mean")
results_df

,n_components,cv_rmsle_mean,cv_rmsle_std,explained_variance_mean
2,50,1.547055,0.035753,0.263991
3,100,1.551203,0.037556,0.370773
1,25,1.567191,0.048740,0.185846
0,10,1.567636,0.052647,0.123227


In [11]:
# Display the same table with compact metric formatting for easier notebook reading.
results_df.style.format(
    {
        "cv_rmsle_mean": "{:.4f}",
        "cv_rmsle_std": "{:.4f}",
        "explained_variance_mean": "{:.4f}",
    }
).hide(axis="index")

n_components,cv_rmsle_mean,cv_rmsle_std,explained_variance_mean
50,1.5471,0.0358,0.2640
100,1.5512,0.0376,0.3708
25,1.5672,0.0487,0.1858
10,1.5676,0.0526,0.1232


## Test Best PCA Setup

In [12]:
# Choose the best component count based on CV, not on the test split.
best_row = results_df.iloc[0]
best_n_components = int(best_row["n_components"])

# Fit scaler and PCA once on the full training split, then transform the held-out test split.
X_train_final, X_test_final, explained_variance = build_pca_features(
    X_train_raw,
    X_test_raw,
    n_components=best_n_components,
)

# Train the final model using only the selected PCA components.
final_model = build_lgbm_regressor(LGBM_PARAMS)
final_model.fit(X_train_final, y_train_log)

# Predict in log space, then invert log1p with expm1 to return to the original target scale.
y_train_pred_log = final_model.predict(X_train_final)
y_train_pred = np.expm1(y_train_pred_log)
# Clipping prevents invalid negative predictions for RMSLE.
y_train_pred = np.clip(y_train_pred, 0, None)

# Repeat the same prediction conversion for the held-out test split.
y_test_pred_log = final_model.predict(X_test_final)
y_test_pred = np.expm1(y_test_pred_log)
y_test_pred = np.clip(y_test_pred, 0, None)

In [13]:
# RMSLE is the primary metric and matches the log-target training objective.
train_rmsle = root_mean_squared_log_error(y_train_raw, y_train_pred)
test_rmsle = root_mean_squared_log_error(y_test_raw, y_test_pred)
# RMSE and MAE are reported on the original target scale for business interpretability.
train_rmse = root_mean_squared_error(y_train_raw, y_train_pred)
test_rmse = root_mean_squared_error(y_test_raw, y_test_pred)
train_mae = mean_absolute_error(y_train_raw, y_train_pred)
test_mae = mean_absolute_error(y_test_raw, y_test_pred)
# R2 gives a familiar variance-explained view, but RMSLE remains the decision metric.
train_r2 = r2_score(y_train_raw, y_train_pred)
test_r2 = r2_score(y_test_raw, y_test_pred)

# Put all final diagnostics into one long table for easy display and report copying.
metrics = pd.DataFrame(
    {
        "metric": [
            "best_n_components",
            "explained_variance",
            "final_features",
            "cv_rmsle_mean",
            "cv_rmsle_std",
            "train_rmsle",
            "test_rmsle",
            "rmsle_gap_test_minus_train",
            "train_rmse",
            "test_rmse",
            "train_mae",
            "test_mae",
            "train_r2",
            "test_r2",
        ],
        "value": [
            best_n_components,
            explained_variance,
            X_train_final.shape[1],
            best_row["cv_rmsle_mean"],
            best_row["cv_rmsle_std"],
            train_rmsle,
            test_rmsle,
            test_rmsle - train_rmsle,
            train_rmse,
            test_rmse,
            train_mae,
            test_mae,
            train_r2,
            test_r2,
        ],
    }
)

# Reuse the project formatter so large currency-like errors and small metrics are readable.
metrics.style.format({"value": format_metric_value}).hide(axis="index")

metric,value
best_n_components,50
explained_variance,0.2424
final_features,50
cv_rmsle_mean,1.5471
cv_rmsle_std,0.0358
train_rmsle,1.0739
test_rmsle,1.5283
rmsle_gap_test_minus_train,0.4544
train_rmse,"6,888,639.6264"
test_rmse,"7,633,920.1968"


In [14]:
# Machine-readable summary of the experiment. Use this when updating final reports.
summary = {
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "model": "LGBMRegressor",
    "pca_components": PCA_COMPONENTS,
    "cv_folds": CV,
    "test_size": TEST_SIZE,
    "seed": SEED,
    "model_params": LGBM_PARAMS,
    "best_n_components": best_n_components,
    "explained_variance": float(explained_variance),
    "final_features": X_train_final.shape[1],
    "cv_rmsle_mean": float(best_row["cv_rmsle_mean"]),
    "cv_rmsle_std": float(best_row["cv_rmsle_std"]),
    "train_rmsle": float(train_rmsle),
    "test_rmsle": float(test_rmsle),
    "rmsle_gap_test_minus_train": float(test_rmsle - train_rmsle),
    "train_rmse": float(train_rmse),
    "test_rmse": float(test_rmse),
    "train_mae": float(train_mae),
    "test_mae": float(test_mae),
    "train_r2": float(train_r2),
    "test_r2": float(test_r2),
}

summary

{'target_transform': 'log1p',
 'primary_metric': 'rmsle',
 'model': 'LGBMRegressor',
 'pca_components': [10, 25, 50, 100],
 'cv_folds': 5,
 'test_size': 0.33,
 'seed': 42,
 'model_params': {'n_estimators': 700,
  'learning_rate': 0.01,
  'num_leaves': 31,
  'max_depth': 6,
  'min_child_samples': 40,
  'subsample': 0.75,
  'subsample_freq': 1,
  'colsample_bytree': 0.75,
  'reg_alpha': 0.1,
  'reg_lambda': 5.0,
  'min_split_gain': 0.05},
 'best_n_components': 50,
 'explained_variance': 0.24239842206749965,
 'final_features': 50,
 'cv_rmsle_mean': 1.5470552374553403,
 'cv_rmsle_std': 0.03575311933894281,
 'train_rmsle': 1.0738980551629964,
 'test_rmsle': 1.5283186252672496,
 'rmsle_gap_test_minus_train': 0.4544205701042532,
 'train_rmse': 6888639.626396867,
 'test_rmse': 7633920.196816016,
 'train_mae': 3575540.0092044943,
 'test_mae': 4363164.881779054,
 'train_r2': 0.3194944541147817,
 'test_r2': 0.0869151843016227}

## How To Read The Result

- Compare `cv_rmsle_mean` against notebook `03` for the original-feature baseline and notebook `08` for the current best model.
- This notebook tests whether compressed PCA components can replace the original anonymous features.
- Accept PCA-only only if CV improves; the test split is a sanity check.

# 10 PCA-Only LGBM Report

## Goal

The goal of this notebook was to test whether PCA components can replace the original processed feature set for the LightGBM model.

## What Was Done

- Loaded `data/processed_data.csv`.
- Used the original processed features as PCA input.
- Used `log1p(target)`.
- Tested PCA component counts: `10`, `25`, `50`, and `100`.
- Fitted `StandardScaler` and `PCA` inside each CV fold to avoid leakage.
- Trained a fixed conservative `LGBMRegressor` on PCA components only.
- Selected the component count by mean 5-fold CV RMSLE.
- Refit scaler, PCA, and LightGBM on the full train split, then evaluated on the held-out test split.

## Main Results

| n_components | CV RMSLE mean | CV RMSLE std | CV explained variance mean |
| ---: | ---: | ---: | ---: |
| 50 | 1.5471 | 0.0358 | 0.2640 |
| 100 | 1.5512 | 0.0376 | 0.3708 |
| 25 | 1.5672 | 0.0487 | 0.1858 |
| 10 | 1.5676 | 0.0526 | 0.1232 |

The best CV candidate was `50` PCA components. In the final train/test fit, those `50` components explained `0.2424` of the train-split variance.

| metric | value |
| --- | ---: |
| Best PCA components | 50 |
| Final features | 50 |
| CV RMSLE mean | 1.5471 |
| CV RMSLE std | 0.0358 |
| Train RMSLE | 1.0739 |
| Test RMSLE | 1.5283 |
| RMSLE gap test-train | 0.4544 |
| Train RMSE | 6,888,639.63 |
| Test RMSE | 7,633,920.20 |
| Train MAE | 3,575,540.01 |
| Test MAE | 4,363,164.88 |
| Train R2 | 0.3195 |
| Test R2 | 0.0869 |

## Conclusion

PCA should not be accepted as a replacement for the original-feature LightGBM setup. The best PCA CV RMSLE is `1.5471`, far worse than the tuned original-feature runs around `1.36`, and the held-out test RMSLE is also weak at `1.5283`. The train/test RMSLE gap of `0.4544` shows severe generalization degradation despite dimensionality reduction. PCA compresses the anonymous features too aggressively for this model: even `100` components explained more variance but did not improve CV RMSLE. This notebook should remain a negative experiment rather than a candidate final model.
